<h1align='center'>合成数据生成和 Unsloth 教程</h1>

## 综合数据生成

在本节中，我们使用 Synthesis-data-kit 中的 CLI 来生成数据集

### 转换为微调格式

此命令使用**另存为**功能将精选的问答对转换为微调格式：
- 从“data/curated/”读取策划的 JSON 文件
- 转换为格式“ft”（使用消息结构微调格式）
- 输出以正确的对话格式保存到“data/final/”
- 生成的格式与标准微调管道兼容

已成功将 2 个文件转换为微调格式。

In [1]:
import json
import glob
from pathlib import Path
from datasets import Dataset

# =====配置=====
data_dir = "./logical_reasoning/data/final"  # 将其更改为您的数据目录

# ===== 第 1 步：查找所有 FT 文件 =====
data_path = Path(data_dir)
ft_files = glob.glob(str(data_path / "*.json"))

# ===== 第 2 步：加载并转换所有文件 =====
all_data = []

for file_path in ft_files:
    # 加载 JSON 文件
    with open(file_path, 'r') as f:
        ft_data = json.load(f)
    
    # 转换每个项目
    for item in ft_data:
        if 'messages' not in item:
            continue
        
        # 仅提取用户和助理消息
        conversation = []
        for msg in item['messages']:
            if msg['role'] == 'user' or msg['role'] == 'assistant':
                conversation.append({
                    "role": msg['role'],
                    "content": msg['content']
                })
        
        # 如果我们至少有一个交易所，请添加到我们的数据中
        if len(conversation) > 0:
            all_data.append({
                "conversations": conversation
            })

print(f"\n🎯 Total conversations: {len(all_data)}")

# ===== 第 3 步：创建 HuggingFace 数据集 =====
dataset = Dataset.from_list(all_data)

# ===== 第 4 步：预览数据 =====
print(json.dumps(dataset[0], indent=2))


🎯 Total conversations: 29
{
  "conversations": [
    {
      "content": "A person is approached by two islanders, Alex and Ben. Alex says, 'Ben is a knave.' Ben says, 'Alex is a knight.' Who is a knight and who is a knave?",
      "role": "user"
    },
    {
      "content": "First, let's assume Alex is a knight. Then, according to Alex's statement, Ben must be a knave. As a knave, Ben's statement 'Alex is a knight' must be false, which creates a contradiction because we assumed Alex is a knight. Therefore, our initial assumption is wrong, and Alex must be a knave. Since Alex is a knave, his statement 'Ben is a knave' is false, meaning Ben is actually a knight. Thus, Alex is a knave, and Ben is a knight.",
      "role": "assistant"
    }
  ]
}


### 加载数据并将其转换为 HuggingFace 数据集

该单元执行全面的数据处理：

1. **查找文件**：定位 `data/final/` 目录中的所有 JSON 文件
2. **加载数据**：读取每个包含微调格式化数据的JSON文件
3. **格式转换**：从微调格式中提取用户和助理消息
4. **构建对话**：使用角色-内容对创建标准化对话格式
5. **Creating Dataset**：将处理后的数据转换为 HuggingFace Dataset 对象

输出显示总共 74 个对话已成功加载和格式化。预览显示了一个示例对话，其中显示了骑士与无赖的逻辑谜题及其解决方案。

## 微调

### 注意：请记住关闭 vLLM 实例！
### 请参阅 https://unsloth.ai/docs/new/unsloth-amd-pytorch-synthetic-data-hackathon#how-do-i-free-amd-gpu-memory

In [2]:
import os
import json
import glob
import torch
import shutil
from pathlib import Path
from datasets import Dataset

### 导入标准库

导入必要的 Python 库以进行微调：
- `os`、`json`、`glob`：文件系统操作和 JSON 处理
- `torch`：PyTorch 深度学习框架
- `shutil`：文件操作
- `Path`：路径操作
- `Dataset`：用于数据处理的 HuggingFace 数据集库

In [3]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, train_on_responses_only
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
#### Unsloth: `hf_xet==1.1.10` and `ipykernel>6.30.1` breaks progress bars. Disabling for now in XET.
#### Unsloth: To re-enable progress bars, please downgrade to `ipykernel==6.30.1` or wait for a fix to
https://github.com/huggingface/xet-core/issues/526
INFO 10-19 03:15:01 [__init__.py:216] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 导入 Unsloth 和训练库

导入专门的库以进行高效的微调：
- Unsloth 的“FastLanguageModel”：优化的模型加载和训练
- `get_chat_template`、`standardize_sharegpt`、`train_on_responses_only`：聊天格式化实用程序
- `SFTConfig`、`SFTTrainer`：来自 TRL 的监督微调配置和训练器
- `DataCollatorForSeq2Seq`：处理序列到序列训练的批处理和填充

### 为无位和字节的 ROCm 设置 Unsloth 模型和分词器

In [4]:
max_seq_length = 1024
dtype = torch.bfloat16  # ROCm 的显式 bfloat16
load_in_4bit = False  

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.3-70B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # ROCm 显式
    trust_remote_code=True,
)

print(f"✅ Loaded: Llama-3.3-70B-Instruct (bfloat16, ROCm compatible)")

# 添加 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # 70B 型号的等级更高
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.6: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.0+rocm631.
   \\   /|    AMD Instinct MI300X VF. Num GPUs = 1. Max memory: 191.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+rocm6.4. ROCm Toolkit: 6.4.43482-0f2d60242. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!
INFO:accelerate.utils.modeling: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Loaded: Llama-3.3-70B-Instruct (bfloat16, ROCm compatible)


Unsloth 2025.10.6 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.


### 使用 LoRA 加载 Llama-3.3-70B 模型

该单元建立了在 AMD ROCm 硬件上进行高效微调的模型：

**型号配置：**
- 型号：Llama-3.3-70B-Instruct（700亿个参数）
- 数据类型：bfloat16，用于 ROCm 兼容性
- 无量化（load_in_4bit=False）以避免位和字节依赖性
- 最大序列长度：1024 个标记

**LoRA（低秩适应）配置：**
- 等级 (r)：64 - 大型 70B 型号的等级更高
- 目标模块：所有注意力层和 MLP 层（q_proj、k_proj、v_proj、o_proj、gate_proj、up_proj、down_proj）
- LoRA阿尔法：64
- 辍学率：0（无辍学）
- 梯度检查点：“不懒惰”以提高内存效率

LoRA 通过仅训练小型适配器层而不是整个 70B 模型来实现高效微调，从而可以在具有 192GB HBM3 内存的单个 AMD MI300X GPU 上进行训练。

In [5]:
"""Prepare dataset with proper chat template and tensor compatibility"""
print("🔧 准备训练数据集...")

# 设置聊天模板
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# 确保设置了填充令牌
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 确保正确张量转换的格式化函数
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    
    for convo in convos:
        # 确保对话格式正确
        if isinstance(convo, list) and all(isinstance(msg, dict) for msg in convo):
            text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        else:
            print(f"⚠️  Skipping malformed conversation: {type(convo)}")
            continue
    
    return {"text": texts}

dataset = standardize_sharegpt(dataset)

dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

print(f"✅ Prepared {len(dataset)} valid examples for training")

# 显示样品
if len(dataset) > 0:
    print(f"📝 Sample formatted text:")
    print(dataset["text"][0][:200] + "...")

🔧 Preparing dataset for training...


Unsloth: Standardizing formats (num_proc=20):   0%|          | 0/29 [00:00<?, ? examples/s]

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

Filter:   0%|          | 0/29 [00:00<?, ? examples/s]

✅ Prepared 29 valid examples for training
📝 Sample formatted text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

A person is approached ...


### 使用聊天模板准备数据集

此单元格格式化数据集以进行微调：

**步骤：**
1. **设置聊天模板**：应用 Llama-3.1 聊天模板格式
2. **配置 Padding**：如果尚未设置，则将 pad 令牌设置为 eos 令牌
3. **格式化对话**：`formatting_prompts_func`函数：
   - 从数据集中获取原始对话
   - 应用聊天模板以正确格式化它们
   - 验证对话结构（具有角色/内容的字典列表）
   - 过滤掉格式错误的对话
4. **标准化格式**：使用`standardize_sharegpt`标准化数据结构
5. **应用格式**：在所有示例中映射格式功能
6. **删除空**：过滤掉任何空的或无效的格式化文本

输出显示已成功准备 74 个有效示例。将显示格式化文本的示例，其中显示了正确的 Llama-3.1 聊天模板结构以及系统、用户和助理标题。

In [6]:
"""Train model with ROCm-optimized settings"""
# 确保分词器具有适当的填充
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 设置训练器，具有 ROCm 友好的设置和正确的数据处理
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=64,  # 🚀 MI300X 可以用 192GB HBM3 处理这个！
        gradient_accumulation_steps=1,   # 有效批量大小 = 8*2 = 16
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=1e-4,
        logging_steps=1,
        optim="adamw_8bit",  # 纯炬优化器
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="logical_reasoning_rocm_outputs",
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        remove_unused_columns=True,  # 删除未使用的列以避免张量问题
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # ROCm 稳定性的单人工作
    ),
)

# 仅根据响应进行训练
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

FastLanguageModel.for_training(model)
trainer_stats = trainer.train()


trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/29 [00:00<?, ? examples/s]

Map (num_proc=24):   0%|          | 0/29 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 29 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 828,375,040 of 71,382,081,536 (1.16% trained)


Step,Training Loss
1,0.866600


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 29 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 828,375,040 of 71,382,081,536 (1.16% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.866600


### 使用 ROCm 优化设置训练模型

该单元配置并执行微调过程：

**训练配置（SFTConfig）：**
- **批量大小**：每台设备 64 个 - 利用 AMD MI300X 的海量 192GB HBM3 内存
- **梯度累积**：1步
- **热身**：5 个步骤
- **Epochs**：1 次完整遍历数据集
- **学习率**：1e-4
- **优化器**：adamw_8bit 提高内存效率
- **精度**：ROCm 的 bf16 (bfloat16)
- **梯度检查点**：启用以提高内存效率

**特殊训练模式：**
使用“train_on_responses_only”仅计算助理响应的损失，而不是用户问题的损失。这使模型专注于学习生成准确的答案，而不是记住输入格式。

**主要特点：**
- DataCollatorForSeq2Seq 使用适当的填充处理可变长度序列
- 无需打包即可保留对话结构
- 用于 ROCm 稳定性的单个数据加载器工作程序
- 通过 Unsloth 进行梯度检查点以实现内存优化

然后模型接受 74 个逻辑推理对话的训练。

In [ ]:
"""Save the trained model"""
print("\n💾 保存 ROCM 训练的模型")

# 保存 LoRA 适配器
lora_path = "logical_reasoning_rocm_lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to: {lora_path}")

# 保存合并模型
merged_path = "logical_reasoning_rocm_merged"
print("🔄 正在保存合并模型...")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to: {merged_path}")

print(f"\n🎉 ROCM MODEL READY!")


💾 SAVING ROCM-TRAINED MODEL
✅ LoRA adapters saved to: logical_reasoning_rocm_lora
🔄 Saving merged model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 30 files from cache to `logical_reasoning_rocm_merged`: 100%|██████████| 30/30 [01:05<00:00,  2.18s/it]


Successfully copied all 30 files from cache to `logical_reasoning_rocm_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:  10%|█         | 3/30 [00:24<03:35,  7.98s/it]

### 保存微调模型

该单元以两种格式保存训练后的模型：

1. **LoRA 适配器** (`逻辑_reasoning_rocm_lora/`)：
   - 仅保存经过训练的 LoRA 适配器权重（轻量级，约几百 MB）
   - 可以稍后与基础模型一起加载
   - 对于与原始基础模型共享或部署非常有用

2. **合并模型** (`logic_reasoning_rocm_merged/`):
   - 将 LoRA 适配器合并回基本模型
   - 创建具有所有权重的独立模型
   - 以 16 位精度保存以获得更好的质量
   - 无需加载适配器即可立即推理

两种格式都包含分词器配置。合并的模型已准备好投入生产，可直接用于生成逻辑推理问题的答案。我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  此笔记本和所有 Unsloth 笔记本均已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>